# Gemini Interactions API 快速開始 (Text Generation)

本 Notebook 示範 Google Gemini 官方推薦的 **Interactions API** (`client.interactions.create`) 語法。
涵蓋基本文字生成、多模態圖文輸入、串流輸出、多輪對話（伺服器端狀態維護）以及思考與系統指示設定。

In [ ]:
from google import genai
import os
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

interaction = client.interactions.create(
    model="gemini-3.7-flash",
    input="AI是如何工作的(請使用繁體中文回答)?"
)

display(Markdown(interaction.output_text))

In [ ]:
from google import genai
from IPython.display import display, Markdown

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.7-flash",
    input="請問你的姓名(請使用繁體中文回答)?"
)

display(Markdown(interaction.output_text))

## 多模態輸入 (Multimodal Inputs)
- 結合文字與圖片輸入產生文字輸出

In [ ]:
import PIL.Image
import io
import base64
from google import genai
from IPython.display import display, Markdown

client = genai.Client()
image = PIL.Image.open("bear.jpg")
buffered = io.BytesIO()
image.save(buffered, format="JPEG")
img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

interaction = client.interactions.create(
    model="gemini-3.7-flash",
    input=[
        {"type": "text", "text": "請告訴我這是什麼動物,還有關於它的一些資訊(請使用繁體中文回答)"},
        {"type": "image", "data": img_b64, "mime_type": "image/jpeg"}
    ]
)

display(Markdown(interaction.output_text))

## 串流回應 (Streaming responses)
- 設定 `stream=True` 並監聽 `step.delta` 取得即時 Token

In [ ]:
from google import genai

client = genai.Client()
stream = client.interactions.create(
    model="gemini-3.7-flash",
    input="AI是如何工作的(請使用繁體中文回答)?",
    stream=True
)
for event in stream:
    if event.event_type == "step.delta" and event.delta.type == "text":
        print(event.delta.text, end="", flush=True)

## 多輪對話 (Multi-turn conversations)
- 使用 `previous_interaction_id` 連結前一輪互動，伺服器端會自動維護對話狀態

In [ ]:
from google import genai
from IPython.display import display, Markdown

client = genai.Client()

# 第一輪對話
interaction1 = client.interactions.create(
    model="gemini-3.7-flash",
    input="我有2隻狗在我的房子內",
)
print("第一輪回答：")
display(Markdown(interaction1.output_text))

# 第二輪對話（傳入 previous_interaction_id）
interaction2 = client.interactions.create(
    model="gemini-3.7-flash",
    input="在我家裏有多少爪子?",
    previous_interaction_id=interaction1.id,
)
print("第二輪回答：")
display(Markdown(interaction2.output_text))

In [ ]:
from google import genai

client = genai.Client()

interaction1 = client.interactions.create(
    model="gemini-3.7-flash",
    input="我有2隻狗在我的房子內",
)
print("第1輪：", interaction1.output_text)

print("\n第2輪（串流）：", end="")
stream = client.interactions.create(
    model="gemini-3.7-flash",
    input="在我家裏有多少爪子?",
    previous_interaction_id=interaction1.id,
    stream=True
)
for event in stream:
    if event.event_type == "step.delta" and event.delta.type == "text":
        print(event.delta.text, end="", flush=True)

## 思考設定與系統指示 (Thinking & System Instructions)
- 透過 `generation_config` 調整 `thinking_level` 或 `temperature`
- 透過 `system_instruction` 引導模型行為

In [ ]:
from google import genai
from IPython.display import display, Markdown

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.7-flash",
    input="說明AI如何工作",
    generation_config={
        "thinking_level": "low",
        "temperature": 1.0
    }
)

display(Markdown(interaction.output_text))

In [ ]:
from google import genai
from IPython.display import display, Markdown

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.7-flash",
    system_instruction="你是一隻貓,你的名字叫Neko.",
    input="早安,您好嗎?"
)

display(Markdown(interaction.output_text))

## 無狀態對話 (Stateless Conversations)
- 當設定 `store=False` 時，伺服器不儲存對話，客戶端可自行傳遞完整的 `steps` 陣列

In [ ]:
from google import genai
from IPython.display import display, Markdown

client = genai.Client()

history = [
    {
        "type": "user_input",
        "content": [{"type": "text", "text": "我有2隻狗在我的房子內"}]
    }
]

interaction1 = client.interactions.create(
    model="gemini-3.7-flash",
    store=False,
    input=history
)
print("第1輪：", interaction1.output_text)

for step in interaction1.steps:
    history.append(step.model_dump())

history.append({
    "type": "user_input",
    "content": [{"type": "text", "text": "在我家裏有多少爪子?"}]
})

interaction2 = client.interactions.create(
    model="gemini-3.7-flash",
    store=False,
    input=history
)
print("第2輪：", interaction2.output_text)